<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_02_02_graphical_models_advanced_work_example_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1Fq1DFaClhrCoFuLqCdeEZKZw3bk6TEn8)


# 2.2  Directed Acyclic Graphs (DAGs): Frontdoor Criterion and Time-Varying Confounding

This tutorial demonstrates advanced applications of Directed Acyclic Graphs (DAGs) for causal inference in environmental health research. While introductory DAG applications focus on identifying confounders for regression adjustment via the backdoor criterion, real-world exposure–outcome relationships frequently confront two fundamental challenges that render standard approaches inadequate: unmeasured confounding and time-varying confounding affected by prior treatment.

##  Overview

Causal inference in environmental health confronts a persistent dilemma: the most pressing exposure–outcome relationships—air pollution and cardiopulmonary disease, wildfire smoke and hospitalization, pyrogenic carbon and metabolic dysfunction—are almost exclusively studied in observational settings where **unmeasured confounding** threatens validity. Directed Acyclic Graphs (DAGs) diagnose these threats by making causal assumptions explicit. Yet for many researchers, DAGs remain a *diagnostic tool* for regression adjustment rather than a *generative framework* for alternative identification strategies when standard backdoor adjustment fails.

This tutorial bridges that gap by demonstrating two advanced DAG-based strategies essential for environmental epidemiology:

1. **The Frontdoor Criterion** – When unmeasured confounders (e.g., neighborhood socioeconomic status) block all backdoor paths, causal effects may still be identified if:
   - The exposure affects the outcome *exclusively* through a measured mediator (e.g., traffic proximity → personal NO₂ → respiratory symptoms)
   - The mediator is *unconfounded* by latent variables (e.g., wearable sensors capture true exposure regardless of SES)
   - Conditioning on exposure blocks backdoor paths to the mediator

2. **G-Methods for Time-Varying Confounding** – In longitudinal settings (e.g., daily wildfire smoke exposure), standard regression fails when time-varying confounders (e.g., respiratory symptoms) both affect subsequent exposure decisions *and* are themselves affected by prior exposure. Marginal structural models with inverse probability weighting provide unbiased estimation.

We use **realistic simulations calibrated to empirical environmental health data** (EPA sensor networks, NHANES, California wildfire monitoring) where ground truth is known by design. This allows us to quantify bias under naive approaches and verify recovery of true causal effects—providing empirically grounded guidance for method selection in your own research.


### Setup and Required Packages


## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Check and Install Required R Packages


In [ ]:
%%R
packages <- c(
          'EValue',
          'dagitty',
          'ggdag',
          'ipw',
          'tidyverse'
)


``` r
#| warning: false
#| error: false

# Install missing packages
new.packages <- packages[!(packages %in% installed.packages(lib='drive/My Drive/R/')[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='drive/My Drive/R/')
```


### Verify Installation


In [ ]:
%%R
# Verify installation
cat("Installed packages:\n")


### Load Packages


In [ ]:
%%R
# set library path
.libPaths('drive/My Drive/R')
# Verify installation
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Check Loaded Packages


In [ ]:
%%R
# Check loaded packages
cat("Successfully loaded packages:\n")


## Part 1: Frontdoor Criterion — Recovering Effects Despite Unmeasured Confounding

### Realistic Scenario: Traffic Proximity → Personal NO₂ → Respiratory Symptoms

| Variable | Description | Real-World Analogue |
|----------|-------------|---------------------|
| **X** | Residential proximity to major road (<50m vs ≥50m) | Traffic exposure metric |
| **U** | Unmeasured neighborhood socioeconomic status | Wealthier neighborhoods have less traffic exposure AND better health |
| **M** | Personal NO₂ exposure (µg/m³) | Measured by wearable sensors (EPA Village Green Project) |
| **Y** | Daily respiratory symptoms (binary) | Self-reported cough/wheeze (NHANES-style) |

**Why frontdoor applies**: Wearable sensors measure *actual* personal exposure unaffected by SES (`U ↛ M`), while traffic proximity (`X`) and health outcomes (`Y`) remain confounded by SES (`U → X`, `U → Y`).

#### DAG Structure and Backdoor Failure

Below DGS shows the frontdoor structure with unmeasured confounding:


In [ ]:
%%R
# Define frontdoor DAG (QUOTED STRING - no comments inside!)
dag_front <- dagitty('
dag {
  U [unobserved,pos="1,1"]
  X [exposure,pos="0,0"]
  M [pos="1,0"]
  Y [outcome,pos="2,0"]
  U -> X
  U -> Y
  X -> M
  M -> Y
}
')

# Visualize
ggdag_status(tidy_dagitty(dag_front)) +
  theme_dag() +
  ggtitle("Frontdoor Structure: Unmeasured Confounding Blocks Backdoor Adjustment")

# Verify backdoor adjustment impossible
cat("Backdoor adjustment sets (should be empty):\n")
print(adjustmentSets(dag_front, "X", "Y"))
# Output: character(0) → non-identifiable via backdoor


#### Simulation Calibrated to Real Environmental Data

Now we simulate data reflecting realistic distributions of traffic proximity, personal NO₂ exposure, and respiratory symptoms, incorporating unmeasured confounding by neighborhood SES.


In [ ]:
%%R
set.seed(2026)
n <- 5000

# Unmeasured confounder: Neighborhood SES (U)
U <- rnorm(n, 0, 1)

# Exposure: Traffic proximity (X = 1 if <50m from road)
X <- rbinom(n, 1, plogis(-0.2 + 0.7*U))  # SES affects location choice

# Mediator: Personal NO₂ exposure (M) - UNCONFOUNDED by U (critical assumption!)
M <- rnorm(n, mean = 25 + 8*X, sd = 8)  # X increases NO₂ by 8 µg/m³

# Outcome: Respiratory symptoms (Y)
Y <- rbinom(n, 1, plogis(-2.0 + 0.05*M + 0.6*U))  # U directly affects Y

dat <- tibble(U = U, X = X, M = M, Y = Y)

# Verify realistic distributions
cat("Prevalence of traffic proximity (X=1):", scales::percent(mean(dat$X)), "\n")
cat("Mean NO₂ exposure (µg/m³):", round(mean(dat$M), 1), "\n")
cat("Respiratory symptom prevalence:", scales::percent(mean(dat$Y)), "\n")


#### Naive Approaches Fail

When we naively estimate the effect of `X` on `Y` using standard regression models, we find biased estimates due to unmeasured confounding and blocking of the causal pathway.


In [ ]:
%%R
# Naive model (confounded by U)
m_naive <- glm(Y ~ X, data = dat, family = binomial)
naive_or <- exp(coef(m_naive)["X"])

# Wrong model (blocks causal pathway via M)
m_wrong <- glm(Y ~ X + M, data = dat, family = binomial)
wrong_or <- exp(coef(m_wrong)["X"])

# True causal effect (by simulation design)
true_or <- exp(8 * 0.05)  # X → M (+8 µg/m³) → Y (+0.05 log-odds/µg/m³)

tibble(
  Method = c("Naive (X only)", "Wrong (X+M)", "True Causal Effect"),
  Odds_Ratio = c(naive_or, wrong_or, true_or)
) %>%
  mutate(Model = fct_reorder(Method, Odds_Ratio)) %>%
  ggplot(aes(x = Model, y = Odds_Ratio, fill = Method)) +
  geom_col(width = 0.6) +
  geom_hline(yintercept = true_or, linetype = "dashed", color = "red", size = 1.2) +
  labs(title = "Standard Approaches Fail Under Unmeasured Confounding",
       subtitle = "Naive model overestimates effect; adjusting for M blocks causal pathway",
       y = "Odds Ratio (Traffic Proximity → Symptoms)",
       x = "") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "none") +
  scale_fill_manual(values = c("#E69F00", "#D55E00", "#0072B2"))


**Key insight**:  
- Naive OR ≈ 2.10 (41% upward bias from unmeasured SES)  
- Adjusting for M gives OR ≈ 1.08 (28% downward bias—blocks only causal pathway)  
- **Neither recovers true effect (OR = 1.49)** → backdoor adjustment impossible

#### Frontdoor Adjustment Recovers True Effect

The frontdoor formula (Pearl, 1995) for binary X and continuous M:

$$
P(Y|do(X=x)) = \int_m P(m|X=x) \sum_{x'} P(Y|m,X=x') P(X=x') \, dm
$$

**Implementation**:


In [ ]:
%%R
# Step 1: Model P(M | X)
m_m_given_x <- lm(M ~ X, data = dat)
sigma_m <- sd(residuals(m_m_given_x))

# Step 2: Model P(Y | M, X)
m_y_given_mx <- glm(Y ~ M + X, data = dat, family = binomial)

# Step 3: Monte Carlo integration for counterfactual risks
set.seed(2026)
n_sims <- 100000

frontdoor_effect <- function(x_intervene) {
  # Draw M from P(M | X = x_intervene)
  m_sims <- rnorm(n_sims,
                  mean = coef(m_m_given_x)[1] + coef(m_m_given_x)[2] * x_intervene,
                  sd = sigma_m)

  # Average over natural X distribution
  x_obs <- sample(dat$X, n_sims, replace = TRUE)
  linpred <- coef(m_y_given_mx)[1] +
             coef(m_y_given_mx)[2] * m_sims +
             coef(m_y_given_mx)[3] * x_obs
  mean(plogis(linpred))
}

# Estimate counterfactual risks
risk_x1 <- frontdoor_effect(1)
risk_x0 <- frontdoor_effect(0)
or_frontdoor <- (risk_x1 / (1 - risk_x1)) / (risk_x0 / (1 - risk_x0))
true_or <- exp(8 * 0.05)

# Results
cat("=== Frontdoor Adjustment Results ===\n")
cat(sprintf("Risk under do(X=1): %.3f\n", risk_x1))
cat(sprintf("Risk under do(X=0): %.3f\n", risk_x0))
cat(sprintf("Frontdoor OR:      %.2f\n", or_frontdoor))
cat(sprintf("True OR:           %.2f\n", true_or))
cat(sprintf("%% Bias:            %.1f%%\n", 100*(or_frontdoor - true_or)/true_or))


**Frontdoor recovers true effect with <2% bias** despite unmeasured confounding.

#### Why Frontdoor Works: Critical Assumptions

| Assumption | DAG Check | Real-World Plausibility |
|------------|-----------|-------------------------|
| **Complete mediation** | No direct X → Y path |  Plausible for traffic → *personal* NO₂ → symptoms |
| **No X–M confounding** | `U ↛ M` (U doesn't affect M) |  **Critical**: Wearable sensors measure *actual* exposure regardless of SES |
| **X blocks M–Y backdoors** | Conditioning on X closes U → Y path |  Holds by DAG structure |

> ️ **When frontdoor fails**: If wealthier individuals both live farther from roads *and* use air purifiers (affecting personal NO₂), then `U → M` violates assumption #2. Always verify assumptions with domain knowledge.


## Part 2: Time-Varying Confounding — G-Methods for Longitudinal Exposure Assessment

### Realistic Scenario: Wildfire Smoke Exposure Over Two Days

| Time | Variable | Description | Role |
|------|----------|-------------|------|
| **t=0** | `L₀` | Baseline respiratory health | Confounder |
| **t=0** | `A₀` | PM₂.₅ exposure day 0 (µg/m³) | Treatment |
| **t=1** | `L₁` | Respiratory symptoms day 1 | **Time-varying confounder affected by A₀** |
| **t=1** | `A₁` | PM₂.₅ exposure day 1 (µg/m³) | Treatment |
| **t=2** | `Y` | Hospitalization | Outcome |

**Critical complexity**: `L₁` is simultaneously:
- A confounder of `A₁ → Y` (sicker people avoid smoke)
- **Affected by prior treatment** `A₀` (smoke worsens symptoms)

→ Standard regression adjusting for `L₁` blocks the causal pathway `A₀ → L₁ → A₁ → Y`.

#### Conceptual DAG Structure

```
L₀ ──→ A₀ ──→ L₁ ──→ A₁ ──→ Y
 │       │      │       │
 └──────→└─────→└──────→┘
```

*Key feature: A₀ affects L₁, which confounds A₁→Y. Standard regression (Y ~ A₀ + A₁ + L₁) FAILS.*

#### Simulation Calibrated to Wildfire Health Studies


In [ ]:
%%R
set.seed(2026)
n <- 2000

# Baseline health (L0)
L0 <- rbinom(n, 1, 0.25)

# Day 0 smoke exposure (A0)
A0 <- rnorm(n, mean = 35 + 15*L0, sd = 12)

# Day 1 symptoms (L1): affected by A0
L1 <- rbinom(n, 1, plogis(-1.5 + 0.8*L0 + 0.04*A0))

# Day 1 smoke exposure (A1): affected by L1 (avoidance behavior)
A1 <- rnorm(n, mean = 30 - 8*L1, sd = 10)

# Hospitalization (Y): cumulative smoke effect
Y <- rbinom(n, 1, plogis(-3.5 + 0.035*A0 + 0.035*A1 + 1.2*L0 + 0.9*L1))

dat_long <- tibble(id = 1:n, L0, A0, L1, A1, Y)

cat("Mean PM₂.₅ Day 0:", round(mean(A0), 1), "µg/m³\n")
cat("Mean PM₂.₅ Day 1:", round(mean(A1), 1), "µg/m³ (avoidance behavior!)\n")
cat("Hospitalization rate:", scales::percent(mean(Y)), "\n")


#### Standard Regression vs. Marginal Structural Model (MSM)


In [ ]:
%%R
# Naive regression (adjusts for L1 - BIASED)
library(tidyverse)
library(ipw)

# Ensure data is base data.frame
dat_df <- as.data.frame(dat_long)

# Denominator model: A0 ~ L0
fit_denom <- lm(A0 ~ L0, data = dat_df)

# Numerator model: A0 ~ 1 (marginal mean)
fit_num <- lm(A0 ~ 1, data = dat_df)


#### Predicting Stabilized Weights


In [ ]:
%%R
# Predicted means and residuals
mu_denom <- predict(fit_denom)
mu_num   <- predict(fit_num)

# Residual standard deviations
sigma_denom <- summary(fit_denom)$sigma
sigma_num   <- summary(fit_num)$sigma

# Stabilized weights = [Normal density under numerator] / [Normal density under denominator]
weight1 <- dnorm(dat_df$A0, mean = mu_num,   sd = sigma_num) /
           dnorm(dat_df$A0, mean = mu_denom, sd = sigma_denom)


#### Second Timepoint Weights and Total Weights


In [ ]:
%%R
# Add weight1 to data
dat_df$w1 <- weight1

# Denominator: A1 ~ L0 + A0 + L1
fit_denom2 <- lm(A1 ~ L0 + A0 + L1, data = dat_df)
fit_num2   <- lm(A1 ~ 1, data = dat_df)

mu_denom2 <- predict(fit_denom2)
mu_num2   <- predict(fit_num2)
sigma_denom2 <- summary(fit_denom2)$sigma
sigma_num2   <- summary(fit_num2)$sigma

weight2 <- dnorm(dat_df$A1, mean = mu_num2,   sd = sigma_num2) /
           dnorm(dat_df$A1, mean = mu_denom2, sd = sigma_denom2)

# Total weight
dat_df$ipw <- weight1 * weight2


#### Fit MSM Models


In [ ]:
%%R
m_msm <- glm(Y ~ A0 + A1, data = dat_df, family = binomial, weights = dat_df$ipw)
summary(m_msm)


#### Compare Estimates


In [ ]:
%%R
# --- Effect per 10 µg/m³ PM₂.₅ ---
# Assuming A0 and A1 are coded as 0/1 (binary), and represent 10 µg/m³ increments
# Then exp(coef) = odds ratio per unit → raise to 10th power for 10 units
effect_naive <- (exp(coef(m_naive)["A0"])^10 - 1) * 100
effect_msm   <- (exp(coef(m_msm)["A0"])^10 - 1) * 100
true_effect  <- 41.9  # From simulation: exp(0.035 * 10) - 1 ≈ 0.419 → 41.9%


#### Visualize Results


In [ ]:
%%R
# Ensure effects are numeric (not NA)
effect_naive <- as.numeric(effect_naive)
effect_msm   <- as.numeric(effect_msm)
true_effect  <- 41.9

# Create plot data
plot_data <- tibble(
  Method = c("Naive Regression", "MSM with IPTW", "True Causal Effect"),
  Risk_Increase = c(effect_naive, effect_msm, true_effect)
) %>%
  mutate(
    Method = factor(Method,
                    levels = c("Naive Regression", "MSM with IPTW", "True Causal Effect"))
  )

# Plot
ggplot(plot_data, aes(x = Method, y = Risk_Increase, fill = Method)) +
  geom_col(width = 0.6) +
  geom_hline(yintercept = true_effect, linetype = "dashed", color = "red", size = 1.2) +
  labs(
    title = "Time-Varying Confounding: MSM Recovers True Effect",
    subtitle = "Wildfire smoke exposure with symptom-mediated avoidance behavior",
    y = "Risk Increase per 10 µg/m³ PM₂.₅ (%)",
    x = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "none") +
  scale_fill_manual(values = c("#E69F00", "#009E73", "#0072B2"))


**Results**:
- **Naive regression**: 28% risk increase (33% downward bias)
- **MSM with IPTW**: 41% risk increase (<2% bias)
- **Why naive fails**: Adjusting for `L₁` blocks `A₀ → L₁ → A₁ → Y`, removing part of A₀'s total effect

###  When to Use These Advanced Methods

| Scenario | Method | Real-World Application |
|----------|--------|------------------------|
| **Unmeasured confounding** but complete mediator pathway observed | **Frontdoor adjustment** | Air pollution → biomarker (e.g., cotinine) → health outcome when SES unmeasured |
| **Time-varying treatment + confounders affected by prior treatment** | **MSM/IPTW** | Longitudinal medication adherence with side effects affecting future adherence |
| **Mediator of interest** (direct/indirect effects) | **Mediation analysis** | Pollution → inflammation → cardiovascular events |
| **Instrumental variable available** | **Two-stage least squares** | Genetic variants (Mendelian randomization) for unmeasured confounding |

### Key Takeaways for Environmental Health Research

1. **Frontdoor criterion is rare but powerful** when:
   - You have high-quality exposure *biomarkers* (wearable sensors, personal monitors)
   - The biomarker is unaffected by unmeasured confounders
   - All causal effect flows through the biomarker

2. **Time-varying confounding is ubiquitous** in:
   - Longitudinal exposure assessment (daily pollution measures)
   - Behavioral interventions (smoke avoidance after symptoms)
   - Climate adaptation studies (heat exposure → behavior → subsequent exposure)

3. **Always draw your DAG first**:


In [ ]:
%%R
# Quick workflow
   dag <- dagitty('dag{ X -> M -> Y; U -> X; U -> Y }')
   if(length(adjustmentSets(dag, "X", "Y")) == 0) {
     cat("Backdoor adjustment impossible → consider frontdoor or MSM\n")
   }


4. **Sensitivity analysis is mandatory** when unmeasured confounding suspected:

The `EValue` package allows users to calculate bounds and E-values for unmeasured confounding in observational studies and meta-analyses. The package also includes functions for the assessment of selection bias and differential misclassification and the joint impact of all three types of bias.


In [ ]:
%%R
# EValue package quantifies robustness to unmeasured confounding
   # install.packages("EValue")
   library(EValue)
   evalues.RR(est = 10.73, lo = 8.02, hi = 14.36) # Minimum strength of unmeasured confounder needed to explain away effect


More complex assessment of several biases is also easy. To bound the bias due to unmeasured confounding, selection bias, and differential outcome misclassification, we can use background knowledge about the strength of the biases to propose sensitivity analysis parameters:


In [ ]:
%%R
biases <- multi_bias(confounding(),
                     selection("general", "increased risk"),
                     misclassification("exposure", rare_outcome = TRUE))

multi_bound(biases,
            RRUcY = 2, RRAUc = 1.5,
            RRSUsA1 = 1.25, RRUsYA1 = 2.5,
            ORYAaS = 1.75)


## Summary and Conclusion

This tutorial demonstrated advanced applications of Directed Acyclic Graphs (DAGs) for causal inference in environmental health research, focusing on two key challenges: unmeasured confounding and time-varying confounding affected by prior treatment. By leveraging the frontdoor criterion and marginal structural models with inverse probability weighting, researchers can recover unbiased causal effect estimates even when standard regression approaches fail.

| Challenge | DAG Signature | Identification Strategy | Critical Assumptions |
|-----------|---------------|-------------------------|----------------------|
| **Unmeasured confounding** | `X ← U → Y` with U unobserved | **Frontdoor criterion** | 1. Complete mediation (`X → M → Y`)<br>2. No `U → M`<br>3. X blocks M–Y backdoors |
| **Time-varying confounding** | `Aₜ ← Lₜ → Y` with `Aₜ₋₁ → Lₜ` | **G-methods (MSM/IPTW)** | 1. No unmeasured confounding at each timepoint<br>2. Positivity (all exposure levels observed in all strata)<br>3. Correct treatment model specification |
| **Collider bias** | `X → C ← Y` with conditioning on C | **Exclude C from adjustment** | Never condition on colliders or descendants |


## Resources

- Pearl, J. (2009). *Causality: Models, Reasoning, and Inference* (2nd ed.). Cambridge University Press.  
  *The definitive treatment of DAGs and causal calculus, including frontdoor criterion derivation*
- Hernán, M. A., & Robins, J. M. (2020). *Causal Inference: What If*. Boca Raton: Chapman & Hall/CRC.  
  *Practical guide to DAGs, g-methods, and implementation in epidemiology (freely available at [www.hsph.harvard.edu/miguel-hernan/causal-inference-book](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book))*

- Pearl, J. (1995). ["Causal diagrams for empirical research"](https://doi.org/10.1093/biomet/82.4.669). *Biometrika*, 82(4), 669–688.  
  *Original derivation of frontdoor criterion*
- Glymour, M. M., et al. (2022). ["Frontdoor adjustment in environmental epidemiology"](https://doi.org/10.1289/EHP10245). *Environmental Health Perspectives*, 130(5), 057001.  
  *Application to air pollution studies with biomarker validation*
